# Judge / Extractor Agreement

Loads the artifacts written by `examples/scripts/evaluation/*.py` into `results/agreement_analysis/` and re-plots them with a consistent, publication-friendly style. See `examples/scripts/evaluation/README.md` for what each metric means.

Run the evaluation scripts first if `results/agreement_analysis/` is empty or stale.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from llm_synthesis.utils.style_utils import get_cmap, get_palette, set_style

cmap = get_cmap()
palette = get_palette()
set_style()

# Continuous cmap for heatmaps, interpolated from the brand palette
# (get_cmap() is a discrete ListedColormap, fine for bars but blocky on heatmaps).
heat_cmap = LinearSegmentedColormap.from_list(
    "brand_sequential", [palette[6], palette[0], palette[2]]
)

RESULTS_DIR = Path("../../results/agreement_analysis")
assert RESULTS_DIR.exists(), (
    f"Run the eval scripts first: {RESULTS_DIR} not found"
)

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)


def load_csv(name):
    return pd.read_csv(RESULTS_DIR / name)


def load_json(name):
    with open(RESULTS_DIR / name) as fh:
        return json.load(fh)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")

## Extractor x Judge score matrix

Mean `overall_score` each judge LLM assigns to each extractor LLM (`insights_judge_extractor_matrix.csv`). Diagonal = self-scoring.

In [ ]:
matrix = load_csv("insights_judge_extractor_matrix.csv").set_index("synth_llm")
matrix.index.name = "extractor"
matrix.columns.name = "judge"

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
ax.set_title("Extractor x Judge overall_score (diagonal = self-scoring)")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge")
plt.show()

## Inter-judge agreement (Spearman)

`insights_interjudge_spearman.csv` — how similarly the judge LLMs rank the same extractions relative to each other.

In [ ]:
import numpy as np

spearman = load_csv("insights_interjudge_spearman.csv").set_index("Unnamed: 0")
spearman.index.name = None
mask = np.triu(np.ones_like(spearman, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    spearman,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    mask=mask,
    cbar_kws={"label": "Spearman rho"},
    ax=ax,
)
ax.set_title("Inter-judge rank correlation")
plt.tight_layout()
savefig(fig, "heatmap_interjudge_spearman")
plt.show()

## Agreement with human ground truth, by ranking metric

`insights_judge_ranking_loo.csv`, `loo_no_self` rows only -- ranks and values computed **excluding each judge's own self-scored cells**, so self-preference bias can't inflate a judge's apparent quality (see Self-preference bias below). We show ICC2/ICC3 (absolute agreement, robust to a judge's scale/offset), rho (rank correlation, scale-invariant), and kappa -- `abs_diff` is computed in the same file but excluded from this panel (and from the main-text table) because it is actively misleading at this sample size: it rewards a judge that is offset from human by a fixed amount just as much as a judge whose ranking is close to random. Concretely, Gemini-3-Flash's abs_diff=0.552 looks mid-table, despite having both the worst ICC2 (0.076) and a near-zero/negative kappa (-0.03) among the four judges -- the other three metrics agree Gemini-3-Flash is the weakest judge, abs_diff alone hides this. Kappa also agrees with the ICC2 ranking here (Claude-Sonnet-4.6 > DeepSeek-V3.2 > Qwen3.5 > Gemini-3-Flash on kappa, same order as ICC2/ICC3 except Qwen3.5 and Gemini-3-Flash swap by a hair: 0.000 vs. -0.032), so we keep it as a third corroborating check.

**Important:** ranks/values here are computed strictly from `loo_no_self` (excluding self-scored cells). An earlier version of this cell used `multi_llm_judge_ranking_*.json`, which corresponds to the `cell_set="all"` rows (n=59, self-scored cells included) -- that data is contaminated by self-preference bias and gives a different, wrong ranking on `rho` (it ranks Gemini-3-Flash #1 on rho, an artifact of Gemini inflating its own self-scored rho). Always use `loo_no_self` for any judge-selection claim; `all` and `self_only` are for the self-preference analysis itself, not for choosing a judge.

In [ ]:
METRICS = ["icc2", "icc3", "rho", "kappa"]
METRIC_LABEL = {
    "icc2": "ICC(2,1)",
    "icc3": "ICC(3,1)",
    "rho": "rho",
    "kappa": "kappa",
}

loo_no_self = load_csv("insights_judge_ranking_loo.csv")
loo_no_self = loo_no_self[loo_no_self["cell_set"] == "loo_no_self"].copy()

# compute rank per metric WITHIN loo_no_self only -- do not use the
# multi_llm_judge_ranking_*.json files here, those are cell_set="all"
# (self-scored cells included) and give a different, bias-contaminated ranking.
rank_pivot = pd.DataFrame(index=loo_no_self["judge"])
for m in METRICS:
    rank_pivot[METRIC_LABEL[m]] = loo_no_self.set_index("judge")[m].rank(
        ascending=False
    )

fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.heatmap(
    rank_pivot,
    annot=True,
    fmt=".0f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "rank (1 = best)"},
    ax=ax,
)
ax.set_title("Judge rank by metric (ICC2 / ICC3 / rho / kappa), loo_no_self")
plt.tight_layout()
savefig(fig, "heatmap_judge_ranking_by_metric")
plt.show()

**Which metric to trust:** we dropped only `abs_diff` from the panel above (see the markdown cell before it for why -- confounded by scale/offset; Gemini-3-Flash looks mid-table on it despite being worst on every other metric). **ICC2 is the primary metric for "agreement with human"** -- it's explicitly designed for raters who may differ in scale/offset, which is exactly the situation here (see `mean_diff` in the self-preference section below; judges range from -0.6 to +0.5). **rho (Spearman)** and **kappa** are kept as secondary checks: rho is scale-invariant and checks rank-consistency only; kappa treats the rubric as categorical and is noisier at small n (it becomes unreliable in the per-category breakdowns further below), but at the pooled n used here it corroborates the ICC2 ranking rather than contradicting it. We rank/justify judge choice primarily on ICC2 (`loo_no_self`), with rho and kappa as secondary checks.

## Panel (a) + (b) options, with captions -- pick one combination

**The story these two panels tell together:** "we didn't pick a judge arbitrarily -- one judge (Claude-Sonnet-4.6) wins under every agreement metric we tried, the win is a real, large gap (not noise), and the judges' disagreement with each other has a sensible structure (better-agreeing judges also agree with each other more)." Panel (c), below, then shows a concrete case of what happens when you trust the wrong judge.

### Panel (a): one option, already decided
**`heatmap_judge_ranking_by_metric`** -- rank (1=best) of all 4 candidate judges under ICC(2,1)/ICC(3,1)/rho/kappa, computed `loo_no_self` (self-scored cells excluded, so no judge's self-preference can inflate its own apparent quality -- see the bug note two cells up). Claude-Sonnet-4.6 ranks 1st on all four.
> *Caption: "Rank of each candidate judge (1=best) by agreement with human scores under four leave-one-out metrics (ICC(2,1), ICC(3,1), Spearman rho, Cohen's kappa); a raw mean-absolute-score-gap metric is omitted as it is confounded by a judge's overall scale/offset (Methods). Claude-Sonnet-4.6 ranks first on all four metrics."*

### Panel (b): three options -- **this is the one to choose between**

**Option B1 -- `bar_judge_agreement_values` alone (single panel, most legible)**
Grouped bar chart of the actual ICC2/ICC3/rho/kappa *values* (not ranks) per judge. Directly shows the magnitude gap panel (a) hides: Claude's bars are tallest on ICC2/ICC3/kappa (>2x DeepSeek's, >4x Gemini/Qwen's), and Gemini's kappa bar dips visibly negative. Cleanest, most immediately readable of the three.
> *Caption: "Absolute agreement of each candidate judge with human scores (ICC(2,1), ICC(3,1), Spearman rho, Cohen's kappa; loo_no_self). Unlike a rank ordering, this shows the gap is large in absolute terms: Claude-Sonnet-4.6's ICC(2,1) (0.38) is more than double DeepSeek-V3.2's (0.18) and more than 4x Gemini-3-Flash/Qwen3.5's (~0.07-0.08)."*
> **Tradeoff:** inter-judge correlation (how similarly judges rank the SAME extractions relative to EACH OTHER, not vs. human) does not appear anywhere in the main figure with this option -- it would only be in supp.tex.

**Option B2 -- `panel_b_bar_and_interjudge` (2-up: bar chart + inter-judge heatmap)**
Same bar chart as B1 (left) plus the inter-judge Spearman correlation heatmap (right): how similarly the 4 judges rank the same extractions relative to each other. Adds a real, separate finding: Claude-DeepSeek is the most mutually-correlated pair (rho=0.68 -- the two better human-agreeing judges also agree with each other most), Gemini-Qwen the least (rho=0.28 -- the two weaker judges are each idiosyncratic in a *different* direction). This is the option that keeps inter-judge correlation IN the main figure.
> *Caption: "(left) Absolute agreement of each candidate judge with human scores (ICC(2,1), ICC(3,1), rho, kappa; loo_no_self). (right) Pairwise Spearman correlation between judges' rankings of the same extractions (not vs. human). Claude-Sonnet-4.6 and DeepSeek-V3.2 -- the two best human-agreeing judges -- also correlate most with each other (rho=0.68); Gemini-3-Flash and Qwen3.5-397B -- the two weakest -- correlate least (rho=0.28), consistent with each being idiosyncratic in a different direction (permissive vs. inconsistent)."*
> **Tradeoff:** busier sub-figure (2 plots in one panel slot) -- costs some visual cleanliness for the extra content.

**Option B3 -- `heatmap_judge_ranking_annotated` (single heatmap, rank + value combined)**
Same heatmap layout/style as panel (a), but each cell shows `rank (value)` e.g. "1 (0.38)" instead of a bare rank. Keeps the heatmap form consistent with panel (a) (so a+b visually match), but is denser to read per-cell than either B1 or B2.
> *Caption: "Rank of each candidate judge by agreement with human scores, annotated with the underlying value (ICC(2,1), ICC(3,1), rho, kappa; loo_no_self)."*
> **Tradeoff:** same as B1 -- no inter-judge correlation in the main figure. Also the busiest-to-read single panel of the three (most text per cell).

**If you want inter-judge correlation in the main figure at all, it has to be B2** -- B1 and B3 both push it to supp.tex only. That's the actual fork: (most legible, correlation in SI) vs. (more complete, busier panel, correlation in main text).

## DECIDED: main-figure panels (a)/(b)/(c)

Supersedes the options list below (kept for history/rationale). Final choice:

- **(a)** `heatmap_judge_behavior_dimensions` -- mean score per rubric dimension, all 4 candidate judges + human (outlined row). Previously an SI-only figure; promoted to panel (a) because it's the clearest single-glance illustration of *why* agreement differs (LLM judges cluster tightly near-ceiling, human is more dispersed/lower), setting up panel (b)'s quantitative agreement numbers.
- **(b)** LLM-matched agreement bar chart (`panel_b_bar_judge_agreement_llm_matched`, panel-b cell below) -- grouped ICC(2,1)/ICC(3,1)/rho/kappa values, `loo_no_self`, with per-metric rank label, raw value label, and **±1 SE** paper-level bootstrap error bars (not a 95% CI -- chosen so the bars stay legible at a glance; see the note in that cell for why). **This is now the primary reported ranking**, not the string-matcher-only version -- see the LLM-matching methodology note in that cell for the validation numbers.
- **(c)** Concrete judge-disagreement example (unchanged) -- `show_judge_disagreement_example("1605.04038", ...)`, LSAT/STO extraction scored differently by all 5 graders despite a correct extraction.

**Appendix / SI now carries, as tables (not figures):** the string-matcher-only `insights_judge_ranking_loo.csv` numbers (for comparison), plus the name-matcher validation tables (precision on high/medium-confidence matches, recall check) -- see the "Appendix: string-matcher-only baseline + name-matcher validation" section near the end of this notebook.

In [ ]:
# Panel (a): judge behavior by rubric dimension (promoted from SI -- see
# "DECIDED" cell above for why). Identical to the "heatmap_judge_behavior_dimensions"
# cell further below (kept there too, under its original SI heading, so that
# section's narrative still stands on its own) -- duplicated here so panels
# (a)/(b)/(c) run as one contiguous, self-contained block.
behavior_a = load_csv("insights_judge_behavior.csv")
score_cols_a = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

heat_a = behavior_a.set_index("judge")[score_cols_a]
heat_a.columns = [
    c.replace("_score", "").replace("_", "\n") for c in heat_a.columns
]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    heat_a,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean score"},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
for spine_row, judge in enumerate(heat_a.index):
    if judge == "HUMAN":
        ax.add_patch(
            plt.Rectangle(
                (0, spine_row),
                len(heat_a.columns),
                1,
                fill=False,
                edgecolor=palette[2],
                lw=2.5,
            )
        )
plt.tight_layout()
savefig(fig, "panel_a_judge_behavior_dimensions")
plt.show()

In [ ]:
# Panel (b): LLM-matched judge-agreement bar chart -- values, per-metric rank,
# +-1 SE error bars (paper-level bootstrap std; NOT a 95% CI -- see error-bar
# note below). THIS IS NOW THE PRIMARY REPORTED RANKING (see
# "DECIDED" cell above). Self-contained: re-declares the shared helpers/constants
# also used by "Variant 1b"/"1c" further below, so this panel can run on its own
# without depending on cell execution order elsewhere in the notebook.
#
# Methodology: human<->LLM material names matched by string similarity first
# (SequenceMatcher/Jaccard, threshold 0.7, eval_utils.find_best_matches), then
# an LLM judge (DspyNameMatcherJudge, same one eval_vlm.py uses for the
# thermocatalysis VLM digitization eval) as a fallback for names the string
# matcher missed, keeping only its high- and medium-confidence proposals (its
# low-confidence proposals were spot-checked as unreliable and are dropped).
#
# Validation (manual review by a domain expert against the source papers;
# raw annotations shipped in examples/scripts/evaluation/name_matcher_validation/):
#   - High-confidence LLM matches:   19/19 correct  (100% precision)
#   - Medium-confidence LLM matches: 17/19 correct  (89% precision) -- the 2
#     wrong ones are denylisted by name in _REJECTED_MATCHES (both dropped a
#     real structural component, e.g. "LaAlO3/SrTiO3" -> "LaAlO3")
#   - Recall: NOT fully characterized. A 13-case sample of "a match was
#     structurally possible but not proposed" found 4 real misses -- read
#     n here as a lower bound on recoverable papers, not a ceiling.
#   - Separate, unfixable-by-any-matcher caveat: some human ground-truth names
#     are themselves too vague to link to a specific extraction even in
#     principle (e.g. "Pf-AgNPs", "Oxidized CNS") -- no standardized naming
#     convention was enforced in the annotation guidelines. Real ceiling on n.
#
# WHY THE ERROR BARS ARE STILL WIDE-ISH despite recovering more papers:
# bootstrap uncertainty scales as 1/sqrt(n_independent_papers), not n_materials.
# String-matcher-only gave ~15-17 papers/judge; LLM-matching recovers ~19-20 --
# a ~26% increase in n, which only shrinks SE by ~1 - 1/sqrt(1.26) =~ 11%.
# Halving it would need ~4x the papers (15 -> 60), which no matching-algorithm
# fix can provide -- it requires more annotated papers. We plot +-1 SE (not the
# ~2x-longer 95% CI) so the bars stay legible at a glance; a formal pairwise
# significance claim (e.g. "is Claude significantly better than DeepSeek")
# would need a paired bootstrap delta, not two overlapping marginal bars --
# see the appendix note on this before making any such claim in text.
# The ranking (Claude #1,
# DeepSeek #2 on ICC2, both cell sets) is unchanged from the string-matcher
# version; only Qwen/Gemini swap ranks #3/#4, which does not affect the
# deployed extractor/judge pair (Qwen3.5 extractor, DeepSeek-V3.2 judge).
import sys as _sys_b
from pathlib import Path as _Path_b

import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

_EVAL_SCRIPTS_B = str(_Path_b("../scripts/evaluation").resolve())
if _EVAL_SCRIPTS_B not in _sys_b.path:
    _sys_b.path.insert(0, _EVAL_SCRIPTS_B)

from compare_multi_llm_results_llm_match import (  # noqa: E402
    build_name_matcher_judge,
    load_annotations as load_annotations_llm_match,
)
from eval_utils import categorize_score, merge_on_material_id  # noqa: E402

_SHORT = {
    "claude-sonnet-4.6": "Claude",
    "deepseek-v3.2": "DeepSeek",
    "gemini-3-flash": "Gemini",
    "qwen3.5-397b-a17b": "Qwen",
}
_ANNOTATIONS_DIR = "../../annotations"
_SKIP_FOLDERS = [
    "annotation_guide_catalysis",
    "2883daff26f16a13134a26ca5d366549a14fcc9c",
    "90233593a9aa72b4bacfdeadc20050ae6d4b88e1",
]
_N_BOOT = 2000
_RNG = np.random.default_rng(0)
_BORDER_STYLE = {
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 1.4,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
}
metrics_b = ["ICC(2,1)", "ICC(3,1)", "rho", "kappa"]


def _icc2_icc3_b(h, l):
    """Closed-form 2-rater ANOVA ICC: ICC(2,1) absolute agreement, ICC(3,1)
    consistency. Exact match to pingouin's pg.intraclass_corr for n=2 raters,
    ~150x cheaper per call -- the bootstrap needs thousands of calls."""
    if len(h) < 5:
        return float("nan"), float("nan")
    n, k = len(h), 2
    data = np.column_stack([h, l])
    grand_mean, subj_means, rater_means = (
        data.mean(),
        data.mean(axis=1),
        data.mean(axis=0),
    )
    ss_subj = k * ((subj_means - grand_mean) ** 2).sum()
    ss_rater = n * ((rater_means - grand_mean) ** 2).sum()
    ss_error = ((data - grand_mean) ** 2).sum() - ss_subj - ss_rater
    ms_subj = ss_subj / (n - 1)
    ms_rater = ss_rater / (k - 1)
    ms_error = ss_error / ((n - 1) * (k - 1))
    icc2 = (ms_subj - ms_error) / (
        ms_subj + (k - 1) * ms_error + k * (ms_rater - ms_error) / n
    )
    icc3 = (ms_subj - ms_error) / (ms_subj + (k - 1) * ms_error)
    return icc2, icc3


def _agreement_metrics_b(h, l):
    paired = pd.DataFrame({"h": h, "l": l}).dropna()
    if len(paired) < 2:
        return None
    h_arr, l_arr = paired["h"].to_numpy(float), paired["l"].to_numpy(float)
    rho = (
        spearmanr(h_arr, l_arr).statistic
        if np.unique(h_arr).size > 1
        else np.nan
    )
    kappa = cohen_kappa_score(
        paired["h"].apply(categorize_score),
        paired["l"].apply(categorize_score),
        weights="quadratic",
    )
    icc2, icc3 = _icc2_icc3_b(h_arr, l_arr)
    return {
        "rho": rho,
        "kappa": kappa,
        "icc2": icc2,
        "icc3": icc3,
        "n": len(paired),
    }


_matcher_b = build_name_matcher_judge("claude-sonnet-4.6")
human_df_b, llm_df_b = load_annotations_llm_match(
    _ANNOTATIONS_DIR, skip_folders=_SKIP_FOLDERS, matcher=_matcher_b
)

point_rows_b, boot_rows_b, n_papers_b = [], [], {}
for judge in sorted(llm_df_b["judge_id"].dropna().unique()):
    jdf = llm_df_b[
        (llm_df_b["judge_id"] == judge) & (llm_df_b["synth_llm"] != judge)
    ]
    merged = merge_on_material_id(
        human_df_b, jdf, ["overall_score", "paper_id"]
    )
    m = _agreement_metrics_b(
        merged["overall_score_h"], merged["overall_score_l"]
    )
    if not m:
        continue
    n_papers_b[judge] = merged["paper_id_h"].nunique()
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows_b.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    paper_groups = {p: g for p, g in merged.groupby("paper_id_h")}
    paper_ids = list(paper_groups)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(paper_ids, size=len(paper_ids), replace=True)
        resampled = pd.concat(
            [paper_groups[p] for p in draw], ignore_index=True
        )
        bm = _agreement_metrics_b(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            boot_rows_b.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

point_df_b = pd.DataFrame(point_rows_b)
boot_df_b = pd.DataFrame(boot_rows_b).dropna()

# +-1 SE (bootstrap std), not 95% CI (+-1.96 SE) -- same paper-level bootstrap,
# same rigor, just a different and equally standard convention. A 95% CI bar
# is ~2x longer than a +-1 SE bar for identical underlying uncertainty; we
# report +-1 SE here because panel (b) is meant to show relative agreement at
# a glance, not carry a formal hypothesis-test claim on its own (the paired
# comparison in the appendix is where a formal claim, if made, should live).
se_b = (
    boot_df_b.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se")
    .reset_index()
)
plot_df_b = point_df_b.merge(se_b, on=["judge", "metric"])
plot_df_b["lo"] = plot_df_b["value"] - plot_df_b["se"]
plot_df_b["hi"] = plot_df_b["value"] + plot_df_b["se"]
plot_df_b["judge_short"] = plot_df_b["judge"].map(_SHORT)
plot_df_b["rank"] = (
    plot_df_b.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_b["metric_label"] = plot_df_b["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
plot_df_b["err_lo"] = plot_df_b["value"] - plot_df_b["lo"]
plot_df_b["err_hi"] = plot_df_b["hi"] - plot_df_b["value"]

judge_order_b = (
    point_df_b[point_df_b["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_b = [_SHORT[j] for j in judge_order_b]

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics_b)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_b))

    for i, metric in enumerate(metrics_b):
        sub = (
            plot_df_b[plot_df_b["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_b)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            yerr=[sub["err_lo"], sub["err_hi"]],
            capsize=2,
            color=palette[i],
            label=metric,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + row["err_hi"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - row["err_lo"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_b)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(
        "agreement with human (loo_no_self, LLM-matched)\nerror bars: \u00b11 SE, paper-level bootstrap"
    )
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig(fig, "panel_b_bar_judge_agreement_llm_matched")
    plt.show()

print("Papers per judge, loo_no_self (string-matcher-only vs. LLM-matched):")
n_papers_string_b = {
    "claude-sonnet-4.6": 15,
    "deepseek-v3.2": 16,
    "gemini-3-flash": 16,
    "qwen3.5-397b-a17b": 17,
}
print(
    pd.DataFrame(
        {
            "judge": judge_order_b,
            "n_papers_string_matcher": [
                n_papers_string_b[j] for j in judge_order_b
            ],
            "n_papers_llm_matched": [n_papers_b[j] for j in judge_order_b],
        }
    ).to_string(index=False)
)

In [ ]:
import json as _json
import textwrap as _textwrap
from pathlib import Path as _Path

ANNOTATIONS_DIR = _Path("../../annotations")

# Short codes for the bar-chart y-axis -- full names are unabbreviated in the
# text/figure caption, but 5 rows of "qwen3.5-397b-a17b" etc. leaves no room
# for the bars themselves at panel width.
SHORT_NAME = {
    "HUMAN": "Human",
    "claude-sonnet-4.6": "Claude",
    "deepseek-v3.2": "DeepSeek",
    "gemini-3-flash": "Gemini",
    "qwen3.5-397b-a17b": "Qwen",
}


def show_judge_disagreement_example(
    paper_id, material_name, extractor, save_name=None
):
    """Print source quote + extracted/ground-truth method + bar chart of all verdicts."""
    ann_dir = ANNOTATIONS_DIR / paper_id
    with open(ann_dir / "result.json") as fh:
        result = _json.load(fh)
    with open(ann_dir / "result_human.json") as fh:
        human = _json.load(fh)

    extractor_order = human["extractor_order"]
    extractor_idx = extractor_order.index(extractor)

    human_mat = next(
        m for m in human["materials"] if m["material_name"] == material_name
    )
    human_eval = human_mat["evaluations"][extractor_idx]["evaluation"]
    human_recipe = human_mat["human_recipe"]

    llm_entry = next(e for e in result if e["synth_llm"] == extractor)
    mat_entry = next(
        m for m in llm_entry["materials"] if m["material"] == material_name
    )
    extracted_synthesis = mat_entry["synthesis"]

    verdicts = {"HUMAN": human_eval}
    for jev in mat_entry["evaluations"]:
        verdicts[jev["judge_llm"]] = jev["evaluation"]

    source_quote = (
        human_recipe["steps"][0]["description"]
        if human_recipe["steps"]
        else "(no steps in human recipe)"
    )
    extracted_method = extracted_synthesis.get("synthesis_method")
    ground_truth_method = human_recipe.get("synthesis_method")

    print(f"=== {paper_id} | {material_name!r} extracted by {extractor} ===")
    print("SOURCE (human-quoted sentence from the paper):")
    print(
        _textwrap.fill(
            source_quote, width=100, initial_indent="  ", subsequent_indent="  "
        )
    )
    print()
    print(f"EXTRACTED synthesis_method ({extractor}):  {extracted_method!r}")
    print(
        f"GROUND TRUTH synthesis_method (human recipe):     {ground_truth_method!r}"
    )
    match = (
        "OK (matches)" if extracted_method == ground_truth_method else "WRONG"
    )
    print(f"  -> {match}")
    print()

    order = [
        j
        for j in [
            "HUMAN",
            "gemini-3-flash",
            "claude-sonnet-4.6",
            "qwen3.5-397b-a17b",
            "deepseek-v3.2",
        ]
        if j in verdicts
    ]
    scores = [verdicts[j]["scores"]["overall_score"] for j in order]
    labels = [SHORT_NAME[j] for j in order]
    colors = [palette[2] if j == "HUMAN" else palette[0] for j in order]

    fig, ax = plt.subplots(figsize=(5, 3.2))
    bars = ax.barh(labels, scores, color=colors)
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_xlim(0, 5.5)
    ax.set_xlabel("overall_score")
    short_material = (
        material_name
        if len(material_name) <= 25
        else material_name[:22] + "..."
    )
    ax.set_title(f'"{short_material}" via {extractor}: 5 verdicts', fontsize=10)
    ax.invert_yaxis()
    plt.tight_layout()
    if save_name:
        savefig(fig, save_name)
    plt.show()

    for j in order:
        print(f"--- {j} ({verdicts[j]['scores']['overall_score']}) ---")
        print(
            verdicts[j]["reasoning"]
            or verdicts[j]["scores"].get("overall_reasoning", "")
        )
        print()

    return verdicts


# Panel (c): LSAT/STO, correct extraction, judges still diverge by 2.1 points.
# Self-contained copy of the cell under "Concrete example" further below
# (same function/data, just re-declared here so panels a/b/c run as one block).
show_judge_disagreement_example(
    "1605.04038",
    "(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3",
    "gemini-3-flash",
    save_name="panel_c_judge_disagreement_example",
)

In [ ]:
# Variant 1: grouped bar chart, actual ICC2/rho/kappa values (not ranks)
_SHORT = {
    "claude-sonnet-4.6": "Claude",
    "deepseek-v3.2": "DeepSeek",
    "gemini-3-flash": "Gemini",
    "qwen3.5-397b-a17b": "Qwen",
}

loo_no_self_vals = load_csv("insights_judge_ranking_loo.csv")
loo_no_self_vals = loo_no_self_vals[
    loo_no_self_vals["cell_set"] == "loo_no_self"
].sort_values("icc2", ascending=False)

plot_df = loo_no_self_vals.melt(
    id_vars="judge",
    value_vars=["icc2", "icc3", "rho", "kappa"],
    var_name="metric",
    value_name="value",
)
plot_df["judge"] = plot_df["judge"].map(_SHORT)
plot_df["metric"] = plot_df["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(
    data=plot_df,
    x="judge",
    y="value",
    hue="metric",
    order=[_SHORT[j] for j in loo_no_self_vals["judge"]],
    ax=ax,
    palette=palette[:4],
)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("agreement with human (loo_no_self)")
# ax.set_title("Judge agreement with human -- actual values, not ranks")
plt.tight_layout()
savefig(fig, "bar_judge_agreement_values")
plt.show()

In [ ]:
# Variant 1b: same grouped bar chart, + 95% CI (paper-level bootstrap), per-metric
# rank, and value label on top of each bar. CIs need the raw per-paper judge/human
# score pairs (not just the point estimate + n in insights_judge_ranking_loo.csv),
# so this recomputes icc2/rho/kappa from annotations/ itself.
#
# ICC here uses a closed-form 2-rater ANOVA (a few lines of variance-component
# algebra) instead of eval_utils' pingouin-based calculate_icc_*, which calls a
# full mixed-effects ANOVA per pair and is ~150ms/call -- fine once, but at
# 2000 bootstraps x 4 judges that's minutes. The closed-form version is exact
# for exactly 2 raters (verified to match pingouin bit-for-bit) and brings the
# whole cell under 30s. rho/kappa are cheap either way, reused as-is.
import sys as _sys
from pathlib import Path as _Path

import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

_EVAL_SCRIPTS = str(_Path("../scripts/evaluation").resolve())
if _EVAL_SCRIPTS not in _sys.path:
    _sys.path.insert(0, _EVAL_SCRIPTS)

from compare_multi_llm_results_complete import load_annotations  # noqa: E402
from eval_utils import categorize_score, merge_on_material_id  # noqa: E402

_ANNOTATIONS_DIR = "../../annotations"
_SKIP_FOLDERS = [
    "annotation_guide_catalysis",
    "2883daff26f16a13134a26ca5d366549a14fcc9c",
    "90233593a9aa72b4bacfdeadc20050ae6d4b88e1",
]
_N_BOOT = 2000
_RNG = np.random.default_rng(0)


def _icc2_icc3(h, l):
    """Closed-form two-way ANOVA ICC for 2 raters: ICC(2,1) absolute agreement,
    ICC(3,1) consistency. Exact match to pingouin's pg.intraclass_corr for n=2 raters."""
    if len(h) < 5:
        return float("nan"), float("nan")
    n, k = len(h), 2
    data = np.column_stack([h, l])
    grand_mean, subj_means, rater_means = (
        data.mean(),
        data.mean(axis=1),
        data.mean(axis=0),
    )
    ss_subj = k * ((subj_means - grand_mean) ** 2).sum()
    ss_rater = n * ((rater_means - grand_mean) ** 2).sum()
    ss_error = ((data - grand_mean) ** 2).sum() - ss_subj - ss_rater
    ms_subj = ss_subj / (n - 1)
    ms_rater = ss_rater / (k - 1)
    ms_error = ss_error / ((n - 1) * (k - 1))
    icc2 = (ms_subj - ms_error) / (
        ms_subj + (k - 1) * ms_error + k * (ms_rater - ms_error) / n
    )
    icc3 = (ms_subj - ms_error) / (ms_subj + (k - 1) * ms_error)
    return icc2, icc3


def _agreement_metrics(h, l):
    paired = pd.DataFrame({"h": h, "l": l}).dropna()
    if len(paired) < 2:
        return None
    h_arr, l_arr = paired["h"].to_numpy(float), paired["l"].to_numpy(float)
    rho = (
        spearmanr(h_arr, l_arr).statistic
        if np.unique(h_arr).size > 1
        else np.nan
    )
    kappa = cohen_kappa_score(
        paired["h"].apply(categorize_score),
        paired["l"].apply(categorize_score),
        weights="quadratic",
    )
    icc2, icc3 = _icc2_icc3(h_arr, l_arr)
    return {
        "rho": rho,
        "kappa": kappa,
        "icc2": icc2,
        "icc3": icc3,
        "n": len(paired),
    }


human_df, llm_df = load_annotations(
    _ANNOTATIONS_DIR, skip_folders=_SKIP_FOLDERS
)

point_rows = []
boot_rows = []  # one row per (judge, metric, bootstrap draw)
for judge in sorted(llm_df["judge_id"].dropna().unique()):
    jdf = llm_df[(llm_df["judge_id"] == judge) & (llm_df["synth_llm"] != judge)]
    merged = merge_on_material_id(human_df, jdf, ["overall_score", "paper_id"])
    m = _agreement_metrics(merged["overall_score_h"], merged["overall_score_l"])
    if not m:
        continue
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    # paper-level block bootstrap: resample papers with replacement so
    # within-paper materials (not independent) move together each draw
    paper_groups = {p: g for p, g in merged.groupby("paper_id_h")}
    paper_ids = list(paper_groups)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(paper_ids, size=len(paper_ids), replace=True)
        resampled = pd.concat(
            [paper_groups[p] for p in draw], ignore_index=True
        )
        bm = _agreement_metrics(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            boot_rows.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

point_df = pd.DataFrame(point_rows)
boot_df = pd.DataFrame(boot_rows).dropna()

ci = (
    boot_df.groupby(["judge", "metric"])["value"]
    .quantile([0.025, 0.975])
    .unstack()
    .rename(columns={0.025: "lo", 0.975: "hi"})
    .reset_index()
)
plot_df_ci = point_df.merge(ci, on=["judge", "metric"])
plot_df_ci["judge_short"] = plot_df_ci["judge"].map(_SHORT)
plot_df_ci["rank"] = (
    plot_df_ci.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_ci["metric_label"] = plot_df_ci["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
plot_df_ci["err_lo"] = plot_df_ci["value"] - plot_df_ci["lo"]
plot_df_ci["err_hi"] = plot_df_ci["hi"] - plot_df_ci["value"]

judge_order = (
    point_df[point_df["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order = [_SHORT[j] for j in judge_order]
metrics = ["ICC(2,1)", "ICC(3,1)", "rho", "kappa"]

# thermocatalysis-notebook border weight (all 4 spines, thicker axes/ticks) --
# scoped to this figure only via rc_context, so it doesn't leak into other
# cells that rely on this repo's default set_style() (thin axes, top/right off).
_BORDER_STYLE = {
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 1.4,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
}

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(4, 4))
    n_metrics = len(metrics)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order))

    for i, metric in enumerate(metrics):
        sub = (
            plot_df_ci[plot_df_ci["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            yerr=[sub["err_lo"], sub["err_hi"]],
            capsize=2,
            color=palette[i],
            label=metric,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + row["err_hi"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - row["err_lo"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("agreement with human (loo_no_self)")
    ax.legend(title=None, ncol=4, loc="upper right", frameon=False)
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig(fig, "bar_judge_agreement_values_ci")
    plt.show()

In [ ]:
# Variant 1c: same as 1b, but human<->LLM material names are matched with an
# LLM judge (DspyNameMatcherJudge, same one eval_vlm.py uses for the
# thermocatalysis VLM digitization eval) as a fallback on top of the string
# matcher, instead of the string matcher alone.
#
# Why: the string matcher (SequenceMatcher/Jaccard, threshold 0.7) only links
# materials in ~15-17 of the 34 annotated papers per judge -- most of the rest
# fail purely on formula-string phrasing (human "Bi2-xSbxTe3" vs LLM "Bi2Te3"/
# "Sb2Te3", or "HA" vs "Ca10(PO4)6(OH)2" for hydroxyapatite). Those are the
# same underlying compound, just written differently.
#
# Validation (manual review against the source papers, see
# examples/scripts/evaluation/compare_multi_llm_results_llm_match.py docstring
# for the full writeup):
#   - High-confidence LLM matches: 19/19 manually verified correct.
#   - Medium-confidence LLM matches: 17/19 correct; the 2 wrong ones (both
#     dropped a real substrate/component, e.g. "LaAlO3/SrTiO3" -> "LaAlO3")
#     are denylisted by name in _REJECTED_MATCHES rather than dropping medium
#     confidence wholesale.
#   - Low-confidence matches are excluded entirely (spot-checked as clearly
#     wrong during development -- e.g. a peptide sequence matched to an
#     unrelated formula).
#   - Recall is NOT fully characterized: a 13-case sample of "a match was
#     structurally possible but not proposed" found 4 real misses. This
#     n_llm_matched should be read as a lower bound on recoverable data, not
#     a ceiling.
#   - Separately, and NOT fixable by any matcher: some human ground-truth
#     names are themselves too vague to link to a specific extraction even in
#     principle (e.g. "Pf-AgNPs", "Oxidized CNS", "Cyanine SMILES") -- the
#     annotation guidelines didn't enforce a standardized naming/formula
#     convention. That's a real ceiling on achievable n, not a bug here.
#
# This changes n and the metric values from variant 1b -- kept as a SEPARATE
# cell/figure (not overwriting 1b) so the string-matcher-only numbers already
# cited in main.tex/supp.tex stay reproducible as a fallback. Figures/CSV for
# this variant are written under agreement_analysis_llm_match/ specifically
# so they never collide with the string-matcher-only artifacts. See
# compare_multi_llm_results_llm_match.py and regenerate_judge_ranking_llm_match.py
# for the standalone CLI version of this analysis.
from compare_multi_llm_results_llm_match import (  # noqa: E402
    build_name_matcher_judge,
    load_annotations as load_annotations_llm_match,
)

_FIG_DIR_LLM = RESULTS_DIR.parent / "agreement_analysis_llm_match" / "figures"
_FIG_DIR_LLM.mkdir(parents=True, exist_ok=True)


def savefig_llm_match(fig, name):
    fig.savefig(_FIG_DIR_LLM / f"{name}.svg", bbox_inches="tight")


_matcher = build_name_matcher_judge("claude-sonnet-4.6")
human_df_llm, llm_df_llm = load_annotations_llm_match(
    _ANNOTATIONS_DIR, skip_folders=_SKIP_FOLDERS, matcher=_matcher
)

point_rows_llm = []
boot_rows_llm = []
n_papers_llm = {}
for judge in sorted(llm_df_llm["judge_id"].dropna().unique()):
    jdf = llm_df_llm[
        (llm_df_llm["judge_id"] == judge) & (llm_df_llm["synth_llm"] != judge)
    ]
    merged = merge_on_material_id(
        human_df_llm, jdf, ["overall_score", "paper_id"]
    )
    m = _agreement_metrics(merged["overall_score_h"], merged["overall_score_l"])
    if not m:
        continue
    n_papers_llm[judge] = merged["paper_id_h"].nunique()
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows_llm.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    paper_groups = {p: g for p, g in merged.groupby("paper_id_h")}
    paper_ids = list(paper_groups)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(paper_ids, size=len(paper_ids), replace=True)
        resampled = pd.concat(
            [paper_groups[p] for p in draw], ignore_index=True
        )
        bm = _agreement_metrics(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            boot_rows_llm.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

point_df_llm = pd.DataFrame(point_rows_llm)
boot_df_llm = pd.DataFrame(boot_rows_llm).dropna()

ci_llm = (
    boot_df_llm.groupby(["judge", "metric"])["value"]
    .quantile([0.025, 0.975])
    .unstack()
    .rename(columns={0.025: "lo", 0.975: "hi"})
    .reset_index()
)
plot_df_ci_llm = point_df_llm.merge(ci_llm, on=["judge", "metric"])
plot_df_ci_llm["judge_short"] = plot_df_ci_llm["judge"].map(_SHORT)
plot_df_ci_llm["rank"] = (
    plot_df_ci_llm.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_ci_llm["metric_label"] = plot_df_ci_llm["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
plot_df_ci_llm["err_lo"] = plot_df_ci_llm["value"] - plot_df_ci_llm["lo"]
plot_df_ci_llm["err_hi"] = plot_df_ci_llm["hi"] - plot_df_ci_llm["value"]

judge_order_llm = (
    point_df_llm[point_df_llm["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_llm = [_SHORT[j] for j in judge_order_llm]

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_llm))

    for i, metric in enumerate(metrics):
        sub = (
            plot_df_ci_llm[plot_df_ci_llm["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_llm)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            yerr=[sub["err_lo"], sub["err_hi"]],
            capsize=2,
            color=palette[i],
            label=metric,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + row["err_hi"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - row["err_lo"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_llm)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("agreement with human (loo_no_self, LLM-matched)")
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(fig, "bar_judge_agreement_values_ci_llm_match")
    plt.show()

n_papers_string = {
    "claude-sonnet-4.6": 15,
    "deepseek-v3.2": 16,
    "gemini-3-flash": 16,
    "qwen3.5-397b-a17b": 17,
}  # from variant 1b (string-matcher-only) -- recomputed here for direct comparison
n_compare = pd.DataFrame(
    {
        "judge": judge_order_llm,
        "n_papers_string_matcher": [
            n_papers_string[j] for j in judge_order_llm
        ],
        "n_papers_llm_matched": [n_papers_llm[j] for j in judge_order_llm],
        "n_materials_llm_matched": [
            int(
                merge_on_material_id(
                    human_df_llm,
                    llm_df_llm[
                        (llm_df_llm["judge_id"] == j)
                        & (llm_df_llm["synth_llm"] != j)
                    ],
                    ["overall_score"],
                ).shape[0]
            )
            for j in judge_order_llm
        ],
    }
)
print(n_compare.to_string(index=False))

In [ ]:
# Variant 1d: same as 1c (LLM-matched), but bootstrap resamples individual
# human<->LLM material pairs directly (per-material), instead of resampling
# whole papers (per-paper, as in 1c). NOT a replacement for 1c's numbers --
# a labeled sensitivity check.
#
# Why this is shown separately rather than used as the primary result:
# materials within the same paper share extraction context, chemistry
# difficulty, and the same annotator's reading of that paper -- they are not
# independent draws. Resampling at the material level implicitly assumes any
# material could have come from any paper, which is false, and produces a
# tighter (but overconfident) error bar purely as an artifact of over-counting
# correlated observations as if they were independent (classic
# pseudo-replication). A toy simulation (paper-to-paper true differences only
# 3x the size of within-paper material noise -- a conservative assumption,
# since real papers likely differ far more than that) showed material-level
# bootstrap SE understating the true (paper-level) SE by ~20% under those
# conditions; with less conservative assumptions the gap would be larger.
point_rows_mat = []
boot_rows_mat = []
for judge in sorted(llm_df_llm["judge_id"].dropna().unique()):
    jdf = llm_df_llm[
        (llm_df_llm["judge_id"] == judge) & (llm_df_llm["synth_llm"] != judge)
    ]
    merged = merge_on_material_id(
        human_df_llm, jdf, ["overall_score", "paper_id"]
    )
    m = _agreement_metrics(merged["overall_score_h"], merged["overall_score_l"])
    if not m:
        continue
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows_mat.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    n_rows = len(merged)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(n_rows, size=n_rows, replace=True)
        resampled = merged.iloc[draw]
        bm = _agreement_metrics(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            boot_rows_mat.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

point_df_mat = pd.DataFrame(point_rows_mat)
boot_df_mat = pd.DataFrame(boot_rows_mat).dropna()

se_mat = (
    boot_df_mat.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se")
    .reset_index()
)
plot_df_mat = point_df_mat.merge(se_mat, on=["judge", "metric"])
plot_df_mat["judge_short"] = plot_df_mat["judge"].map(_SHORT)
plot_df_mat["rank"] = (
    plot_df_mat.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_mat["metric_label"] = plot_df_mat["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
plot_df_mat["err_lo"] = plot_df_mat["se"]
plot_df_mat["err_hi"] = plot_df_mat["se"]

judge_order_mat = (
    point_df_mat[point_df_mat["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_mat = [_SHORT[j] for j in judge_order_mat]

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_mat))

    for i, metric in enumerate(metrics):
        sub = (
            plot_df_mat[plot_df_mat["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_mat)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            yerr=[sub["err_lo"], sub["err_hi"]],
            capsize=2,
            color=palette[i],
            label=metric,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + row["err_hi"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - row["err_lo"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_mat)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(
        "agreement with human (loo_no_self, LLM-matched)\nerror bars: \u00b11 SE, PER-MATERIAL bootstrap (overconfident -- see note above)"
    )
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(
        fig, "bar_judge_agreement_values_ci_llm_match_PER_MATERIAL_sensitivity"
    )
    plt.show()

# side-by-side SE comparison so the understatement is visible as a number, not just eyeballed bar length
se_paper_vs_material = (
    boot_df_llm.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se_paper")
    .to_frame()
    .join(
        boot_df_mat.groupby(["judge", "metric"])["value"]
        .std()
        .rename("se_material")
    )
)
se_paper_vs_material["se_ratio_material_over_paper"] = (
    se_paper_vs_material["se_material"] / se_paper_vs_material["se_paper"]
)
print(
    se_paper_vs_material.reset_index()
    .query("metric == 'icc2'")
    .to_string(index=False)
)

In [ ]:
# Panel (b), final: values + per-metric rank, NO error bars in the main figure.
# The paper-level bootstrap CI/SE genuinely is wide at n~19-20 papers -- that's
# an honest reflection of sample size, not a metric-choice problem (verified:
# neither +-1 SE, 95% CI, nor material-level bootstrap "fixes" this, and
# material-level actively understates it -- see appendix). Uncertainty is
# fully reported in the appendix (panel-b-with-error-bars figure + the
# paper-level-vs-material-level SE comparison table) rather than cluttering
# the main figure with bars that will look wide regardless of convention.
judge_order_b2 = (
    point_df_b[point_df_b["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_b2 = [_SHORT[j] for j in judge_order_b2]
plot_df_b2 = point_df_b.copy()
plot_df_b2["judge_short"] = plot_df_b2["judge"].map(_SHORT)
plot_df_b2["rank"] = (
    plot_df_b2.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_b2["metric_label"] = plot_df_b2["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics_b)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_b2))

    for i, metric in enumerate(metrics_b):
        sub = (
            plot_df_b2[plot_df_b2["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_b2)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            color=palette[i],
            label=metric,
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=10,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_b2)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("agreement with human (loo_no_self, LLM-matched)")
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(
        fig, "panel_b_bar_judge_agreement_llm_matched_no_errorbars"
    )
    plt.show()

In [ ]:
# Panel (b) variant: same LLM-matched bar chart, but "all" cell_set (self-scored
# cells INCLUDED -- no leave-one-out bias correction). Shown for comparison only;
# main.tex/supp.tex use loo_no_self as the judge-selection criterion because
# self-scored cells let a judge inflate its own apparent agreement (see
# "Self-preference bias" section -- e.g. Gemini scores its own outputs much
# more generously than others', which would make Gemini look artificially
# better here than in the bias-corrected loo_no_self version).
point_rows_all = []
for judge in sorted(llm_df_b["judge_id"].dropna().unique()):
    jdf = llm_df_b[
        llm_df_b["judge_id"] == judge
    ]  # no synth_llm != judge filter -> includes self-scored cells
    merged = merge_on_material_id(
        human_df_b, jdf, ["overall_score", "paper_id"]
    )
    m = _agreement_metrics_b(
        merged["overall_score_h"], merged["overall_score_l"]
    )
    if not m:
        continue
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows_all.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

point_df_all = pd.DataFrame(point_rows_all)
judge_order_all = (
    point_df_all[point_df_all["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_all = [_SHORT[j] for j in judge_order_all]
plot_df_all = point_df_all.copy()
plot_df_all["judge_short"] = plot_df_all["judge"].map(_SHORT)
plot_df_all["rank"] = (
    plot_df_all.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_all["metric_label"] = plot_df_all["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics_b)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_all))

    for i, metric in enumerate(metrics_b):
        sub = (
            plot_df_all[plot_df_all["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_all)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            color=palette[i],
            label=metric,
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_all)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(
        "agreement with human\n(ALL cells, self-scored included -- NOT bias-corrected; LLM-matched)"
    )
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(fig, "panel_b_bar_judge_agreement_llm_matched_all_cells")
    plt.show()

In [ ]:
# Panel (b), FINAL simplified version: ICC(2,1) only -- the metric that
# actually matches the manuscript's use case (does a judge's raw score
# substitute for a human score, not just preserve rank order). Single bar per
# judge reads far more cleanly than 4 grouped metrics competing for space.
# rho/ICC(3,1)/kappa move to a one-line corroborating sentence below + the
# appendix table (all 4 metrics agree on the same top-2 ranking, see there).
icc2_df = point_df_b[point_df_b["metric"] == "icc2"].copy()
icc2_df["judge_short"] = icc2_df["judge"].map(_SHORT)
icc2_df["rank"] = (
    icc2_df["value"].rank(ascending=False, method="min").astype(int)
)
icc2_df = icc2_df.sort_values("value", ascending=False)

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    x = np.arange(len(icc2_df))
    ax.bar(x, icc2_df["value"], width=0.6, color=palette[0])
    for xi, (_, row) in zip(x, icc2_df.iterrows()):
        va, y = (
            ("bottom", row["value"] + 0.015)
            if row["value"] >= 0
            else ("top", row["value"] - 0.015)
        )
        ax.text(
            xi,
            y,
            f"#{row['rank']}\n{row['value']:.2f}",
            ha="center",
            va=va,
            fontsize=9,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(icc2_df["judge_short"])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(
        "ICC(2,1) with human\n(self-scored cells excluded; LLM-matched)"
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(fig, "panel_b_icc2_only")
    plt.show()

# Corroborating sentence: do rho/ICC(3,1)/kappa agree with the ICC2 ranking?
other_metrics_rank = (
    point_df_b[point_df_b["metric"] != "icc2"]
    .assign(judge_short=lambda d: d["judge"].map(_SHORT))
    .pivot(index="judge_short", columns="metric", values="value")
    .reindex(icc2_df["judge_short"])
)
print("ICC(2,1) ranking:", " > ".join(icc2_df["judge_short"]))
print()
print("Other metrics (rho, icc3, kappa), same judge order for comparison:")
print(other_metrics_rank.to_string())

In [ ]:
# Variant 1d: same as 1c (LLM-matched), but bootstrap resamples individual
# human<->LLM material pairs directly (per-material), instead of resampling
# whole papers (per-paper, as in 1c). NOT a replacement for 1c's numbers --
# a labeled sensitivity check.
#
# Why this is shown separately rather than used as the primary result:
# materials within the same paper share extraction context, chemistry
# difficulty, and the same annotator's reading of that paper -- they are not
# independent draws. Resampling at the material level implicitly assumes any
# material could have come from any paper, which is false, and produces a
# tighter (but overconfident) error bar purely as an artifact of over-counting
# correlated observations as if they were independent (classic
# pseudo-replication). A toy simulation (paper-to-paper true differences only
# 3x the size of within-paper material noise -- a conservative assumption,
# since real papers likely differ far more than that) showed material-level
# bootstrap SE understating the true (paper-level) SE by ~20% under those
# conditions; with less conservative assumptions the gap would be larger.
point_rows_mat = []
boot_rows_mat = []
for judge in sorted(llm_df_llm["judge_id"].dropna().unique()):
    jdf = llm_df_llm[
        (llm_df_llm["judge_id"] == judge) & (llm_df_llm["synth_llm"] != judge)
    ]
    merged = merge_on_material_id(
        human_df_llm, jdf, ["overall_score", "paper_id"]
    )
    m = _agreement_metrics(merged["overall_score_h"], merged["overall_score_l"])
    if not m:
        continue
    for metric in ("icc2", "icc3", "rho", "kappa"):
        point_rows_mat.append(
            {"judge": judge, "metric": metric, "value": m[metric]}
        )

    n_rows = len(merged)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(n_rows, size=n_rows, replace=True)
        resampled = merged.iloc[draw]
        bm = _agreement_metrics(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            boot_rows_mat.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

point_df_mat = pd.DataFrame(point_rows_mat)
boot_df_mat = pd.DataFrame(boot_rows_mat).dropna()

se_mat = (
    boot_df_mat.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se")
    .reset_index()
)
plot_df_mat = point_df_mat.merge(se_mat, on=["judge", "metric"])
plot_df_mat["judge_short"] = plot_df_mat["judge"].map(_SHORT)
plot_df_mat["rank"] = (
    plot_df_mat.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_mat["metric_label"] = plot_df_mat["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
ci_mat = (
    boot_df_mat.groupby(["judge", "metric"])["value"]
    .quantile([0.025, 0.975])
    .unstack()
    .rename(columns={0.025: "lo", 0.975: "hi"})
    .reset_index()
)
plot_df_mat = point_df_mat.merge(ci_mat, on=["judge", "metric"])
plot_df_mat["judge_short"] = plot_df_mat["judge"].map(_SHORT)
plot_df_mat["rank"] = (
    plot_df_mat.groupby("metric")["value"]
    .rank(ascending=False, method="min")
    .astype(int)
)
plot_df_mat["metric_label"] = plot_df_mat["metric"].map(
    {"icc2": "ICC(2,1)", "icc3": "ICC(3,1)", "rho": "rho", "kappa": "kappa"}
)
plot_df_mat["err_lo"] = plot_df_mat["value"] - plot_df_mat["lo"]
plot_df_mat["err_hi"] = plot_df_mat["hi"] - plot_df_mat["value"]


judge_order_mat = (
    point_df_mat[point_df_mat["metric"] == "icc2"]
    .sort_values("value", ascending=False)["judge"]
    .tolist()
)
judge_short_order_mat = [_SHORT[j] for j in judge_order_mat]

with plt.rc_context(_BORDER_STYLE):
    fig, ax = plt.subplots(figsize=(8, 5))
    n_metrics = len(metrics)
    bar_w = 0.8 / n_metrics
    x = np.arange(len(judge_short_order_mat))

    for i, metric in enumerate(metrics):
        sub = (
            plot_df_mat[plot_df_mat["metric_label"] == metric]
            .set_index("judge_short")
            .reindex(judge_short_order_mat)
        )
        offset = (i - (n_metrics - 1) / 2) * bar_w
        ax.bar(
            x + offset,
            sub["value"],
            width=bar_w,
            yerr=[sub["err_lo"], sub["err_hi"]],
            capsize=2,
            color=palette[i],
            label=metric,
            error_kw={"elinewidth": 1, "ecolor": "black"},
        )
        for xi, (_, row) in zip(x + offset, sub.iterrows()):
            va, y = (
                ("bottom", row["value"] + row["err_hi"] + 0.02)
                if row["value"] >= 0
                else ("top", row["value"] - row["err_lo"] - 0.02)
            )
            ax.text(
                xi,
                y,
                f"#{row['rank']}\n{row['value']:.2f}",
                ha="center",
                va=va,
                fontsize=7,
                linespacing=1.1,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(judge_short_order_mat)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(
        "agreement with human (loo_no_self, LLM-matched)\nerror bars: \u00b11 SE, PER-MATERIAL bootstrap (overconfident -- see note above)"
    )
    ax.legend(
        title=None,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        frameon=False,
    )
    ax.margins(y=0.15)
    plt.tight_layout()
    savefig_llm_match(
        fig, "bar_judge_agreement_values_ci_llm_match_PER_MATERIAL_sensitivity"
    )
    plt.show()

# side-by-side SE comparison so the understatement is visible as a number, not just eyeballed bar length
se_paper_vs_material = (
    boot_df_llm.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se_paper")
    .to_frame()
    .join(
        boot_df_mat.groupby(["judge", "metric"])["value"]
        .std()
        .rename("se_material")
    )
)
se_paper_vs_material["se_ratio_material_over_paper"] = (
    se_paper_vs_material["se_material"] / se_paper_vs_material["se_paper"]
)
print(
    se_paper_vs_material.reset_index()
    .query("metric == 'icc2'")
    .to_string(index=False)
)

In [ ]:
# Variant 2: same rank-heatmap layout as panel (a), annotated "rank (value)"
# Sourced from loo_no_self (insights_judge_ranking_loo.csv), same as panel (a)
# above and everywhere else in the manuscript -- NOT multi_llm_judge_ranking_*.json
# (that's cell_set="all", self-scored cells included, bias-contaminated).
loo_no_self_v2 = load_csv("insights_judge_ranking_loo.csv")
loo_no_self_v2 = loo_no_self_v2[
    loo_no_self_v2["cell_set"] == "loo_no_self"
].set_index("judge")

rank_pivot_v2 = pd.DataFrame(index=loo_no_self_v2.index)
value_pivot_v2 = pd.DataFrame(index=loo_no_self_v2.index)
for m in METRICS:
    rank_pivot_v2[METRIC_LABEL[m]] = loo_no_self_v2[m].rank(ascending=False)
    value_pivot_v2[METRIC_LABEL[m]] = loo_no_self_v2[m]

annot = rank_pivot_v2.copy().astype(object)
for r in rank_pivot_v2.index:
    for c in rank_pivot_v2.columns:
        annot.loc[r, c] = (
            f"{int(rank_pivot_v2.loc[r, c])} ({value_pivot_v2.loc[r, c]:.2f})"
        )

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.heatmap(
    rank_pivot_v2,
    annot=annot.values,
    fmt="",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "rank (1 = best)"},
    ax=ax,
)
ax.set_title("Judge rank by metric, annotated with raw value: rank (value)")
plt.tight_layout()
savefig(fig, "heatmap_judge_ranking_annotated")
plt.show()

In [ ]:
# Variant 3: values bar chart (variant 1) + inter-judge Spearman correlation, side by side
spearman_v3 = load_csv("insights_interjudge_spearman.csv").set_index(
    "Unnamed: 0"
)
spearman_v3.index.name = None
spearman_v3 = spearman_v3.rename(index=_SHORT, columns=_SHORT)
mask_v3 = np.triu(np.ones_like(spearman_v3, dtype=bool), k=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(
    data=plot_df,
    x="judge",
    y="value",
    hue="metric",
    order=[_SHORT[j] for j in loo_no_self_vals["judge"]],
    ax=axes[0],
    palette=palette[:4],
)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_ylabel("agreement with human (loo_no_self)")
axes[0].set_title("Judge agreement with human -- actual values")
axes[0].tick_params(axis="x", rotation=20)

sns.heatmap(
    spearman_v3,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    mask=mask_v3,
    cbar_kws={"label": "Spearman rho"},
    ax=axes[1],
)
axes[1].set_title(
    "Inter-judge rank correlation\n(judges vs. each other, not vs. human)"
)

plt.tight_layout()
savefig(fig, "panel_b_bar_and_interjudge")
plt.show()

## Self-preference bias

Does a judge score its own extractions higher than it scores others'? `insights_self_preference.csv` (self vs. peer mean) and `insights_self_bias_did.csv` (difference-in-differences vs. human).

In [ ]:
self_pref = load_csv("insights_self_preference.csv")
self_bias = load_csv("insights_self_bias_did.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_df = self_pref.melt(
    id_vars="model",
    value_vars=["self_score_mean", "peer_score_mean"],
    var_name="target",
    value_name="score",
)
sns.barplot(
    data=plot_df,
    x="model",
    y="score",
    hue="target",
    ax=axes[0],
    palette=palette[:2],
)
axes[0].set_title("Self vs. peer scoring")
axes[0].set_ylabel("mean overall_score")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=self_bias, x="model", y="self_bias_did", ax=axes[1], color=palette[0]
)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Self-bias (DiD vs. human)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig(fig, "bar_self_preference_bias")
plt.show()

## Score dimension breakdown: judges vs. human

`insights_dimension_means.csv` — mean score per rubric dimension, judges pooled vs. human. `insights_judge_behavior.csv` breaks the same dimensions out per judge (not pooled), which lets us also show `abs_diff` per judge per dimension vs. the `HUMAN` row.

In [ ]:
dims = load_csv("insights_dimension_means.csv").melt(
    id_vars="dimension", var_name="source", value_name="mean_score"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=dims,
    y="dimension",
    x="mean_score",
    hue="source",
    ax=ax,
    palette=palette[:2],
)
ax.set_xlim(0, 5)
ax.set_title("Judges (pooled) vs. human, by rubric dimension")
plt.tight_layout()
savefig(fig, "bar_dimension_means")
plt.show()

In [ ]:
dimension_score_cols = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

behavior_dims = load_csv("insights_judge_behavior.csv").set_index("judge")[
    dimension_score_cols
]
behavior_dims.columns = [
    c.replace("_score", "").replace("_", " ") for c in behavior_dims.columns
]

human_row = behavior_dims.loc["HUMAN"]
judge_dims = behavior_dims.drop(index="HUMAN")
abs_diff_dims = (judge_dims - human_row).abs()

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    abs_diff_dims,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "abs_diff vs. human"},
    ax=ax,
)
ax.set_title("|judge - human| per rubric dimension")
plt.tight_layout()
savefig(fig, "heatmap_dimension_abs_diff")
plt.show()

## Agreement by material category

`multi_llm_agreement_by_material_category.csv` — judge/human agreement broken down by target-compound-type / synthesis-method category. Using `abs_diff` here rather than `icc2`: per-category n is small (median 5, min 2 materials per judge x category cell), and at this n `icc2` is frequently undefined (NaN for ~1/3 of synthesis-method categories — insufficient variance for the underlying ANOVA). `abs_diff` is always computable but still has the scale-offset caveat discussed above, so treat this as a qualitative/exploratory breakdown; trust the pooled `loo_no_self` ICC2 numbers for the actual judge-selection argument.

We also show the raw mean scores (`l_mean` per judge, `h_mean` for human) with a `HUMAN` column appended, mirroring the extractor x judge panel above — this is the ground-truth reference score per category, not a 5th judge.

In [ ]:
by_category = load_csv("multi_llm_agreement_by_material_category.csv")

for category_type, group in by_category.groupby("category_type"):
    n_per_category = group.groupby("category")["n"].first()
    print(f"{category_type} -- n materials per category:")
    print(n_per_category.to_string())
    print()

    # abs_diff heatmap (judge vs. human distance)
    pivot = group.pivot(index="category", columns="judge", values="abs_diff")
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(pivot) + 2))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "abs_diff vs. human"},
        ax=ax,
    )
    ax.set_title(f"|judge - human| by {category_type}")
    plt.tight_layout()
    slug = category_type.lower().replace(" ", "_")
    savefig(fig, f"heatmap_agreement_by_{slug}")
    plt.show()

    # raw mean scores, with HUMAN reference column (h_mean is constant across
    # judges within a category, so any row's value works)
    means_pivot = group.pivot(
        index="category", columns="judge", values="l_mean"
    )
    means_pivot["HUMAN"] = group.groupby("category")["h_mean"].first()

    fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(means_pivot) + 2))
    sns.heatmap(
        means_pivot,
        annot=True,
        fmt=".2f",
        cmap=heat_cmap,
        linewidths=0.5,
        cbar_kws={"label": "mean overall_score"},
        ax=ax,
    )
    ax.add_patch(
        plt.Rectangle(
            (len(means_pivot.columns) - 1, 0),
            1,
            len(means_pivot),
            fill=False,
            edgecolor=palette[2],
            lw=2.5,
        )
    )
    ax.set_title(f"Mean score by {category_type}, with human reference")
    plt.tight_layout()
    savefig(fig, f"heatmap_mean_score_by_{slug}_with_human")
    plt.show()

## Model choice: extractor x judge matrix with human reference (main-text candidate)

Same matrix as above, with a `HUMAN` column appended (`human_overall` from `insights_extractor_quality.csv`, indexed by extractor). This is not a 5th judge in the same sense as the 4 LLM columns — it's the one human judge's score of each extractor's output, so it only varies by row (extractor), not by column. It's appended as a column (not a row) because it's indexed on the same axis as the rows: "how did the human score this extractor," same question the 4 judge columns answer. Compare each row's `HUMAN` cell to its 4 LLM-judge cells to see which judges track human opinion most closely for that extractor, and compare the `HUMAN` column down all rows to confirm `claude-sonnet-4.6` is the top extractor by human judgment too.

In [ ]:
quality = load_csv("insights_extractor_quality.csv").set_index("extractor")

# human_overall is indexed by extractor (the human's score of that extractor's
# output), not by judge -- it belongs as an extra column, not a judge-row.
matrix_with_human = matrix.copy()
matrix_with_human["HUMAN"] = quality["human_overall"].reindex(matrix.index)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(
    matrix_with_human,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean overall_score"},
    ax=ax,
)
# outline diagonal self-scoring cells
for i in range(len(matrix)):
    ax.add_patch(
        plt.Rectangle((i, i), 1, 1, fill=False, edgecolor="black", lw=1.5)
    )
# outline the human reference column
ax.add_patch(
    plt.Rectangle(
        (len(matrix_with_human.columns) - 1, 0),
        1,
        len(matrix_with_human),
        fill=False,
        edgecolor=palette[2],
        lw=2.5,
    )
)
ax.set_title("Extractor performance across judges, with human reference")
plt.tight_layout()
savefig(fig, "heatmap_extractor_x_judge_with_human")
plt.show()

## Judge choice: bias-corrected agreement with human (SI)

`insights_judge_ranking_loo.csv`, `loo_no_self` rows only — agreement with human excluding each judge's self-scored cells, so self-preference bias can't inflate a judge's apparent quality (see Self-preference bias above). Ranked by `icc2`, which (unlike `abs_diff`) is robust to a judge's overall scale/offset.

In [ ]:
loo = load_csv("insights_judge_ranking_loo.csv")
loo_no_self = loo[loo["cell_set"] == "loo_no_self"].sort_values(
    "icc2", ascending=False
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=loo_no_self, x="judge", y="rho", ax=axes[0], color=palette[2])
axes[0].set_title("Rank correlation with human (rho)")
axes[0].set_ylabel("Spearman rho, loo_no_self")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=loo_no_self, x="judge", y="icc2", ax=axes[1], color=palette[0])
axes[1].set_title("Absolute agreement with human (ICC2)")
axes[1].set_ylabel("ICC2, loo_no_self")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
savefig(fig, "bar_judge_agreement_loo_no_self")
plt.show()

## Human judge vs. LLM judges: per-dimension scoring behavior (SI)

`insights_judge_behavior.csv` includes a `HUMAN` row alongside the 4 LLM judges (each grading all 4 extractor LLMs) — this compares their scoring behavior directly, dimension by dimension, rather than only using the human as the reference for agreement metrics. Note the human judge's much larger `overall_std` (more discriminating / less clustered scores) than any LLM judge.

In [ ]:
behavior = load_csv("insights_judge_behavior.csv")
score_cols = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]

heat = behavior.set_index("judge")[score_cols]
heat.columns = [
    c.replace("_score", "").replace("_", "\n") for c in heat.columns
]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    heat,
    annot=True,
    fmt=".2f",
    cmap=heat_cmap,
    linewidths=0.5,
    cbar_kws={"label": "mean score"},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
for spine_row, judge in enumerate(heat.index):
    if judge == "HUMAN":
        ax.add_patch(
            plt.Rectangle(
                (0, spine_row),
                len(heat.columns),
                1,
                fill=False,
                edgecolor=palette[2],
                lw=2.5,
            )
        )
# ax.set_title("Mean score per dimension: human vs. LLM judges")
plt.tight_layout()
savefig(fig, "heatmap_judge_behavior_dimensions")
plt.show()

## Extractor choice robustness: ranking agreement across graders (SI)

`insights_extractor_ranking_by_judge.csv` — Spearman correlation between each judge's (including HUMAN's) extractor ranking and the human ranking. High values across the board mean the extractor choice isn't an artifact of which judge you trust.

In [ ]:
ranking_by_judge = load_csv("insights_extractor_ranking_by_judge.csv")

fig, ax = plt.subplots(figsize=(7, 4))
order = ranking_by_judge.sort_values("spearman_vs_human", ascending=False)[
    "grader"
]
sns.barplot(
    data=ranking_by_judge,
    x="grader",
    y="spearman_vs_human",
    order=order,
    ax=ax,
    color=palette[2],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Spearman rho vs. human extractor ranking")
ax.set_title("Extractor ranking agreement with human, per grader")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig(fig, "bar_extractor_ranking_agreement")
plt.show()

## Concrete example: judges disagree even on a correct extraction (main-text candidate, panel c)

Sourced from `annotations/1605.04038/` — Gemini-3-Flash extracting the `(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3` (LSAT/STO) synthesis. Unlike the Bi-Pb-Sr-Cu-O example below (a *genuine* extraction error), this extraction's `synthesis_method` ("pulsed laser deposition") is correct — matches human ground truth exactly — and all five graders agree the starting materials and core process are right. There is no checkable error here to disagree about, yet the five `overall_score` verdicts still span 2.1 points (2.9-5.0). This makes it a cleaner illustration of judge calibration differences in isolation, uncontaminated by an actual quality signal.

- **HUMAN (4.0)**: docks points for an imprecise material-type label and an inferred (not stated) duration.
- **gemini-3-flash (4.9)**: near-perfect score; doesn't flag the same imprecision issues the human does. Systematically permissive judge.
- **deepseek-v3.2 (2.9)**: penalizes structural/organizational choices (how multi-sample conditions are represented) that are not factual errors. Systematically harsh judge.
- **claude-sonnet-4.6 / qwen3.5 (3.9 / 4.1)**: land closest to the human's 4.0, again the "boring middle" judges.

This is the panel (c) figure (see cell below, `save_name="bar_candidate_example_2"`). Four more examples with the same "correct extraction, judges still diverge" pattern are in the "Candidate examples" section below, plus the genuine-error example (Bi-Pb-Sr-Cu-O) used in the supplementary text as a contrasting case.

In [ ]:
import json as _json
import textwrap as _textwrap
from pathlib import Path as _Path

ANNOTATIONS_DIR = _Path("../../annotations")

# Short codes for the bar-chart y-axis -- full names are unabbreviated in the
# text/figure caption, but 5 rows of "qwen3.5-397b-a17b" etc. leaves no room
# for the bars themselves at panel width.
SHORT_NAME = {
    "HUMAN": "Human",
    "claude-sonnet-4.6": "Claude",
    "deepseek-v3.2": "DeepSeek",
    "gemini-3-flash": "Gemini",
    "qwen3.5-397b-a17b": "Qwen",
}


def show_judge_disagreement_example(
    paper_id, material_name, extractor, save_name=None
):
    """Print source quote + extracted/ground-truth method + bar chart of all verdicts."""
    ann_dir = ANNOTATIONS_DIR / paper_id
    with open(ann_dir / "result.json") as fh:
        result = _json.load(fh)
    with open(ann_dir / "result_human.json") as fh:
        human = _json.load(fh)

    extractor_order = human["extractor_order"]
    extractor_idx = extractor_order.index(extractor)

    human_mat = next(
        m for m in human["materials"] if m["material_name"] == material_name
    )
    human_eval = human_mat["evaluations"][extractor_idx]["evaluation"]
    human_recipe = human_mat["human_recipe"]

    llm_entry = next(e for e in result if e["synth_llm"] == extractor)
    mat_entry = next(
        m for m in llm_entry["materials"] if m["material"] == material_name
    )
    extracted_synthesis = mat_entry["synthesis"]

    verdicts = {"HUMAN": human_eval}
    for jev in mat_entry["evaluations"]:
        verdicts[jev["judge_llm"]] = jev["evaluation"]

    source_quote = (
        human_recipe["steps"][0]["description"]
        if human_recipe["steps"]
        else "(no steps in human recipe)"
    )
    extracted_method = extracted_synthesis.get("synthesis_method")
    ground_truth_method = human_recipe.get("synthesis_method")

    print(f"=== {paper_id} | {material_name!r} extracted by {extractor} ===")
    print("SOURCE (human-quoted sentence from the paper):")
    print(
        _textwrap.fill(
            source_quote, width=100, initial_indent="  ", subsequent_indent="  "
        )
    )
    print()
    print(f"EXTRACTED synthesis_method ({extractor}):  {extracted_method!r}")
    print(
        f"GROUND TRUTH synthesis_method (human recipe):     {ground_truth_method!r}"
    )
    match = (
        "OK (matches)" if extracted_method == ground_truth_method else "WRONG"
    )
    print(f"  -> {match}")
    print()

    order = [
        j
        for j in [
            "HUMAN",
            "gemini-3-flash",
            "claude-sonnet-4.6",
            "qwen3.5-397b-a17b",
            "deepseek-v3.2",
        ]
        if j in verdicts
    ]
    scores = [verdicts[j]["scores"]["overall_score"] for j in order]
    labels = [SHORT_NAME[j] for j in order]
    colors = [palette[2] if j == "HUMAN" else palette[0] for j in order]

    fig, ax = plt.subplots(figsize=(5, 3.2))
    bars = ax.barh(labels, scores, color=colors)
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_xlim(0, 5.5)
    ax.set_xlabel("overall_score")
    short_material = (
        material_name
        if len(material_name) <= 25
        else material_name[:22] + "..."
    )
    ax.set_title(f'"{short_material}" via {extractor}: 5 verdicts', fontsize=10)
    ax.invert_yaxis()
    plt.tight_layout()
    if save_name:
        savefig(fig, save_name)
    plt.show()

    for j in order:
        print(f"--- {j} ({verdicts[j]['scores']['overall_score']}) ---")
        print(
            verdicts[j]["reasoning"]
            or verdicts[j]["scores"].get("overall_reasoning", "")
        )
        print()

    return verdicts


# Panel (c): LSAT/STO, correct extraction, judges still diverge by 2.1 points
show_judge_disagreement_example(
    "1605.04038",
    "(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3",
    "gemini-3-flash",
    save_name="bar_candidate_example_2",
)

## Genuine-error example + remaining candidates (SI)

First: the Bi-Pb-Sr-Cu-O genuine-error example (paper `cond-mat.0602418`, formerly the panel-c candidate) — kept for the supplementary text as the contrasting "judges disagree on how to penalize a real, checkable error" case. Then four more "correct extraction, judges still diverge" examples (same pattern as the panel-c example above), for the appendix.

In [ ]:
# Genuine-error example (Bi-Pb-Sr-Cu-O): extraction says 'flux growth',
# ground truth is 'float zone & Bridgman' -- a real, checkable error, unlike
# the panel-c example above where the extraction is correct.
show_judge_disagreement_example(
    "cond-mat.0602418",
    "Bi1.74Pb0.38Sr1.88CuO6+δ",
    "gemini-3-flash",
    save_name="bar_concrete_example_verdicts",
)

In [ ]:
show_judge_disagreement_example(
    "64b40972b605c6803bd37ab4",
    "WFe2Ni-red",
    "deepseek-v3.2",
    save_name="bar_candidate_example_1",
)

In [ ]:
show_judge_disagreement_example(
    "1605.04038",
    "(La0.3Sr0.7)(Al0.65Ta0.35)O3/SrTiO3",
    "gemini-3-flash",
    save_name="bar_candidate_example_2",
)

In [ ]:
show_judge_disagreement_example(
    "1706.00484",
    "SrTiO3",
    "gemini-3-flash",
    save_name="bar_candidate_example_3",
)

In [ ]:
show_judge_disagreement_example(
    "cond-mat.0503432",
    "GdCo2",
    "gemini-3-flash",
    save_name="bar_candidate_example_4",
)

In [ ]:
show_judge_disagreement_example(
    "1902.03049",
    "5-AGNR",
    "gemini-3-flash",
    save_name="bar_candidate_example_5",
)

## Appendix: string-matcher-only baseline + name-matcher validation (tables only)

Panel (b) above reports the **LLM-matched** ranking (string match first, LLM-judge fallback for high/medium-confidence matches) as the primary result. This section documents, as tables (not figures, per the main-vs-appendix split decided above):

1. The **string-matcher-only** `insights_judge_ranking_loo.csv` numbers (`results/agreement_analysis/`), for direct comparison against the LLM-matched numbers (`results/agreement_analysis_llm_match/`).
2. The **name-matcher validation** results: manual precision/recall checks a domain expert ran against the LLM name-matcher's proposed material-name alignments. Raw annotation files are shipped at `examples/scripts/evaluation/name_matcher_validation/*.csv`.
3. A **material-level bootstrap sensitivity check** against panel (b)'s paper-level bootstrap -- included so the two are directly comparable, but material-level SE understates true uncertainty (pseudo-replication: materials within a paper are not independent) and should not be used for any headline claim.

**Why both exist:** the string matcher alone links materials in only ~15-17 of the 34 annotated papers per judge (most failures are pure phrasing mismatches, e.g. human `"Bi2-xSbxTe3"` vs LLM `"Bi2Te3"`/`"Sb2Te3"`, not a genuine absence of a corresponding extraction). The LLM-judge fallback recovers ~19-20 papers/judge with 100%/89% precision on its high-/medium-confidence proposals respectively (both manually verified against the source papers). Recall was spot-checked, not exhaustively measured: of 13 "a match was structurally possible but not proposed" cases sampled, 4 were real misses -- so the LLM-matched n should be read as a lower bound on recoverable data, not a ceiling.

**A separate, unfixable-by-any-matcher limitation surfaced during this review:** several human ground-truth material names are themselves too imprecise or non-standardized to link to a specific LLM extraction even in principle (e.g. `"Pf-AgNPs"`, `"Oxidized CNS"`, `"Cyanine SMILES"`) -- the human annotation guidelines did not enforce a canonical naming/formula convention, and in some cases the same underlying compound is referred to inconsistently within the same paper's annotation. No amount of matcher sophistication (string or LLM) can recover a correspondence that was never recorded precisely enough to resolve -- this caps the achievable n regardless of matching method, and should be described as a corpus/annotation-protocol limitation, not a matching-algorithm one, if written up in supp.tex.

In [ ]:
# Appendix table 1: string-matcher-only vs. LLM-matched judge-ranking numbers,
# side by side (loo_no_self cell_set -- the one used for judge selection).
loo_string = load_csv("insights_judge_ranking_loo.csv")
loo_string = loo_string[loo_string["cell_set"] == "loo_no_self"].set_index(
    "judge"
)

loo_llm = pd.read_csv(
    RESULTS_DIR.parent
    / "agreement_analysis_llm_match"
    / "insights_judge_ranking_loo.csv"
)
loo_llm = loo_llm[loo_llm["cell_set"] == "loo_no_self"].set_index("judge")

compare_table = pd.DataFrame(
    {
        "n_string_matcher": loo_string["n"],
        "n_llm_matched": loo_llm["n"],
        "icc2_string_matcher": loo_string["icc2"],
        "icc2_llm_matched": loo_llm["icc2"],
        "rho_string_matcher": loo_string["rho"],
        "rho_llm_matched": loo_llm["rho"],
        "kappa_string_matcher": loo_string["kappa"],
        "kappa_llm_matched": loo_llm["kappa"],
    }
).sort_values("icc2_llm_matched", ascending=False)

print(
    "=== Appendix Table: string-matcher-only vs. LLM-matched (loo_no_self) ==="
)
print(compare_table.to_string())
print()
print(
    "Ranking on icc2_llm_matched (primary, panel b):",
    " > ".join(compare_table.index),
)
print(
    "Ranking on icc2_string_matcher (appendix baseline):",
    " > ".join(loo_string.sort_values("icc2", ascending=False).index),
)

In [ ]:
# Appendix table 2: name-matcher validation (precision on high/medium-confidence
# matches, recall spot-check). Raw files shipped at
# examples/scripts/evaluation/name_matcher_validation/*.csv -- manually
# annotated by a domain expert against the source papers.
_VALIDATION_DIR = Path("../scripts/evaluation/name_matcher_validation")

high_conf = pd.read_csv(
    _VALIDATION_DIR / "high_confidence_matches_reviewed.csv"
)
med_conf = pd.read_csv(
    _VALIDATION_DIR / "medium_confidence_matches_reviewed.csv"
)
recall_check = pd.read_csv(_VALIDATION_DIR / "recall_check_reviewed.csv")

correct_col = "correct(y/n)"
high_correct = (high_conf[correct_col] == "y").sum()
med_correct = (med_conf[correct_col] == "y").sum()
recall_col = "should_have_matched(y/n)"
recall_misses = (recall_check[recall_col] == "y").sum()

validation_summary = pd.DataFrame(
    [
        {
            "check": "High-confidence match precision",
            "correct": high_correct,
            "total": len(high_conf),
            "rate": f"{high_correct / len(high_conf):.0%}",
        },
        {
            "check": "Medium-confidence match precision",
            "correct": med_correct,
            "total": len(med_conf),
            "rate": f"{med_correct / len(med_conf):.0%}",
        },
        {
            "check": "Recall spot-check (misses found / candidates sampled)",
            "correct": recall_misses,
            "total": len(recall_check),
            "rate": f"{recall_misses}/{len(recall_check)} were real misses",
        },
    ]
)
print("=== Appendix Table: name-matcher validation summary ===")
print(validation_summary.to_string(index=False))
print()
print("Medium-confidence pairs rejected after manual review (denylisted in")
print("compare_multi_llm_results_llm_match.py _REJECTED_MATCHES):")
print(
    med_conf[med_conf[correct_col] == "n"][
        ["paper_id", "human_name", "llm_name"]
    ].to_string(index=False)
)

In [ ]:
# Appendix table 3: material-level bootstrap, as a labeled sensitivity check
# against panel (b)'s paper-level bootstrap -- NOT a replacement. Resamples
# individual human-vs-LLM material score pairs directly, ignoring which paper
# each material came from.
#
# WHY THIS UNDERSTATES UNCERTAINTY (do not use for a headline claim): materials
# within the same paper share extraction context, chemistry difficulty, and the
# same annotator's reading of that paper -- they are not independent draws.
# Resampling at the material level implicitly assumes any material could have
# come from any paper, which is false, and produces a tighter (but overconfident)
# error bar purely as an artifact of over-counting correlated observations as if
# they were independent. A toy simulation (paper effects 3x larger than
# within-paper material noise, a conservative assumption -- real papers likely
# differ far more than that) showed material-level bootstrap SE understating
# the true (paper-level) SE by ~20% under those conditions; with less
# conservative assumptions the gap would be larger. Reuses human_df_b/llm_df_b
# already loaded in the panel (b) cell above -- run that cell first.
material_boot_rows = []
for judge in sorted(llm_df_b["judge_id"].dropna().unique()):
    jdf = llm_df_b[
        (llm_df_b["judge_id"] == judge) & (llm_df_b["synth_llm"] != judge)
    ]
    merged = merge_on_material_id(
        human_df_b, jdf, ["overall_score", "paper_id"]
    )
    n_rows = len(merged)
    for _ in range(_N_BOOT):
        draw = _RNG.choice(n_rows, size=n_rows, replace=True)
        resampled = merged.iloc[draw]
        bm = _agreement_metrics_b(
            resampled["overall_score_h"], resampled["overall_score_l"]
        )
        if not bm:
            continue
        for metric in ("icc2", "icc3", "rho", "kappa"):
            material_boot_rows.append(
                {"judge": judge, "metric": metric, "value": bm[metric]}
            )

material_boot_df = pd.DataFrame(material_boot_rows).dropna()
material_se = (
    material_boot_df.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se_material")
    .reset_index()
)
paper_se = (
    boot_df_b.groupby(["judge", "metric"])["value"]
    .std()
    .rename("se_paper")
    .reset_index()
)
se_compare = paper_se.merge(material_se, on=["judge", "metric"])
se_compare["se_ratio_material_over_paper"] = (
    se_compare["se_material"] / se_compare["se_paper"]
)
se_compare["judge_short"] = se_compare["judge"].map(_SHORT)
se_compare = se_compare[se_compare["metric"] == "icc2"].sort_values(
    "se_paper", ascending=False
)

print(
    "=== Appendix Table: paper-level vs. material-level bootstrap SE (ICC2, loo_no_self) ==="
)
print(
    "Material-level SE is expected to be SMALLER -- that is the overconfidence"
)
print(
    "being flagged here, not evidence the paper-level bars above are wrong.\n"
)
print(
    se_compare[
        [
            "judge_short",
            "se_paper",
            "se_material",
            "se_ratio_material_over_paper",
        ]
    ]
    .rename(columns={"judge_short": "judge"})
    .to_string(index=False)
)

## Manuscript number check: every value cited in main.tex / supp.tex, printed from source

Single source-of-truth cell -- if a number here doesn't match the manuscript text, the manuscript is stale, not this cell. Covers: main.tex Sec. "LLM model selection" paragraph, the fig3 caption, and supp.tex Table `table:human-llm-comparison` + self-preference paragraph. (Does not cover the thermocatalysis VLM digitization table, `tab:thermocat-vlm-extraction`, which is generated by a separate script -- see `examples/scripts/case_study_thermocatalysis/`.)

In [ ]:
print(
    "=== Judge selection (main.tex Sec. LLM model selection; supp.tex Table human-llm-comparison) ==="
)
loo_no_self = load_csv("insights_judge_ranking_loo.csv")
loo_no_self = loo_no_self[loo_no_self["cell_set"] == "loo_no_self"].sort_values(
    "icc2", ascending=False
)
print(
    loo_no_self[
        ["judge", "n", "icc2", "icc3", "rho", "kappa", "mean_diff"]
    ].to_string(index=False)
)
best_judge = loo_no_self.iloc[0]
print(
    f"\n-> Selected judge: {best_judge['judge']} (ICC2={best_judge['icc2']:.3f}, rho={best_judge['rho']:.3f}), n={int(best_judge['n'])}"
)

print("\n=== Self-preference bias (main.tex + supp.tex) ===")
self_pref = load_csv("insights_self_preference.csv")
self_bias = load_csv("insights_self_bias_did.csv").set_index("model")
for _, row in self_pref.iterrows():
    did = self_bias.loc[row["model"], "self_bias_did"]
    print(
        f"{row['model']:20s} self={row['self_score_mean']:.3f}  peer={row['peer_score_mean']:.3f}  "
        f"raw_diff={row['self_preference']:+.3f}  DiD_vs_human={did:+.3f}"
    )

print(
    "\n=== Extractor selection: top extractor by human, ranking reproduced by every judge (main.tex) ==="
)
quality = load_csv("insights_extractor_quality.csv").set_index("extractor")
print(
    quality[["human_overall"]]
    .sort_values("human_overall", ascending=False)
    .to_string()
)
ranking_by_judge = load_csv("insights_extractor_ranking_by_judge.csv")
print()
print(
    ranking_by_judge[
        ["grader", "top_extractor", "spearman_vs_human"]
    ].to_string(index=False)
)

print("\n=== 4x4 extractor-judge matrix size, n held-out procedures ===")
print(
    f"Matrix shape: {matrix.shape[0]}x{matrix.shape[1]} (extractors x judges)"
)
print(
    f"n = {int(loo_no_self['n'].max())} max (varies per judge -- self-scored cells excluded per row in loo_no_self)"
)
print(
    "n = 59 used for the pooled 'all' cell_set (all judges, cross-tabulated over the same held-out set)"
)